# Can We Predict UFC Fight Outcomes?
### A data-driven look at styles, stats, and matchups

MMA is the hardest mainstream sport to forecast, and it's not particularly close.

One clean shot ends everything. A guy losing every exchange for fourteen minutes can land a check hook in the last round and walk out with a highlight-reel knockout. A heavy favorite slips on a kick, gets dragged to the mat, and taps to a choke from someone the books had at +400. Styles make fights – a world-class wrestler with bad striking is a nightmare for a world-class striker with mediocre takedown defense, even if the striker has the better "record".

And records lie. A 12-2 fighter who built that record on regional shows is not the same as a 12-2 fighter who went 12-2 inside the UFC top 10. Strength of schedule is invisible until you start poking at the numbers.

This notebook is an honest attempt – not a crystal ball. I want to see how far simple, well-engineered features can take us, where they break, and what the upsets actually teach us about the limits of modeling combat sports.

Dataset: [UFC Fight Data – 2026](https://www.kaggle.com/datasets/anthonysz/ufc-fight-data-2026) (fights from 2010 through May 2026, with pre-computed differential features and per-corner career stats).


---
## 1. Data Loading & First Look

Quick rule I follow with any new sports dataset: load it, look at the shape, look at the dtypes, look at what's missing. Don't model anything until you trust what's on the page.


In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

# A darker, octagon-ish palette
PALETTE = ["#d62728", "#1f77b4", "#f5b041", "#2ecc71", "#9b59b6", "#e74c3c"]
RED, BLUE = "#d62728", "#1f77b4"

sns.set_theme(style="darkgrid", rc={
    "figure.facecolor": "#0f1116",
    "axes.facecolor":  "#161a22",
    "axes.edgecolor":  "#2a2f3a",
    "axes.labelcolor": "#d0d0d0",
    "xtick.color":     "#b0b0b0",
    "ytick.color":     "#b0b0b0",
    "text.color":      "#e0e0e0",
    "axes.titlesize":  13,
    "axes.titleweight":"bold",
    "grid.color":      "#23283180",
})
PLOTLY_TEMPLATE = "plotly_dark"

# Resolve dataset path - works on Kaggle and locally
CANDIDATE_DIRS = [
    "/kaggle/input/ufc-fight-data-2026",
    "/kaggle/input/datasets/anthonysz/ufc-fight-data-2026",
    ".",
]
DATA_DIR = next((p for p in CANDIDATE_DIRS if os.path.exists(p)), ".")
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print("Using data dir:", DATA_DIR)
print("CSV files found:", csv_files)


In [ ]:
raw = pd.read_csv(os.path.join(DATA_DIR, "all_fights.csv"))
print("Shape:", raw.shape)
raw.head(3)


In [ ]:
dtype_summary = (
    raw.dtypes.value_counts()
    .rename_axis("dtype").reset_index(name="n_columns")
)
print(dtype_summary.to_string(index=False))
print(f"\nDate range: {raw['fight_date'].min()}  ->  {raw['fight_date'].max()}")
print(f"Unique fighters (red+blue): {pd.unique(raw[['red_fighter','blue_fighter']].values.ravel()).size}")


In [ ]:
missing = raw.isna().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(9, max(3, 0.32 * len(missing))))
sns.barplot(x=missing.values, y=missing.index, color=RED, ax=ax)
ax.set_title("Missing values by column (% of rows)")
ax.set_xlabel("% missing")
plt.tight_layout(); plt.show()

print(f"{len(missing)} columns have at least one missing value.")


**What's actually here, honestly.**

- ~7,260 fights, ~16 years of UFC history, with both raw per-corner stats (`red_*`, `blue_*`) **and** pre-computed differential columns (`*_diff`). That's a gift – we don't have to engineer everything from scratch.
- Betting odds (`red_odds`, `blue_odds`) are included. That's huge. Closing-line odds are the single most informative variable in combat sports prediction, so we need to be careful not to let them dominate everything else.
- The dataset doesn't expose a per-fight **method of victory** column (KO vs Sub vs Decision happened *that night*). What we do have is each fighter's career KO/sub counts at the time of the fight. We'll use those to infer style, but we should be upfront that "win method distribution" here means *career-aggregated finishing tendencies*, not per-bout outcomes.
- Rank columns (`b_match_wc_rank`, `r_match_wc_rank`) and stance fields will have legitimate missingness – unranked fighters, weird stance entries. We'll handle them gently.


---
## 2. Data Cleaning

Light hands here. Aggressive cleaning destroys signal – I'd rather impute conservatively and lose a few fights than over-engineer the inputs.


In [ ]:
df = raw.copy()

# 1. fight outcome -> clean boolean. 't'/'f' is awkward.
df["red_win"] = (df["red_winner"].astype(str).str.lower() == "t").astype(int)

# 2. dates
df["fight_date"] = pd.to_datetime(df["fight_date"], errors="coerce")
df["year"] = df["fight_date"].dt.year

# 3. names
for c in ["red_fighter", "blue_fighter"]:
    df[c] = df[c].astype(str).str.strip()

# 4. weight class - collapse the obvious junk
df["weight_class"] = df["weight_class"].fillna("Unknown").str.strip()
# Catch Weight bouts are real but tiny - keep them but flag
df["is_catchweight"] = (df["weight_class"] == "Catch Weight").astype(int)

# 5. title bout 't'/'f'
df["title_bout"] = (df["title_bout"].astype(str).str.lower() == "t").astype(int)

# 6. stance - keep top categories, bucket the rest
for c in ["red_stance", "blue_stance"]:
    df[c] = df[c].fillna("Unknown").str.strip().str.title()
    df.loc[~df[c].isin(["Orthodox", "Southpaw", "Switch", "Open Stance"]), c] = "Other"

# 7. physical stats - median impute by weight class (preserves division differences)
phys = ["red_height_cms", "red_reach_cms", "blue_height_cms", "blue_reach_cms",
        "red_age", "blue_age"]
before = df[phys].isna().sum().sum()
for c in phys:
    df[c] = df.groupby("weight_class")[c].transform(lambda s: s.fillna(s.median()))
    df[c] = df[c].fillna(df[c].median())  # safety net
after = df[phys].isna().sum().sum()
print(f"Physical stat NaNs: {before} -> {after}")

# 8. ranks: NaN means unranked. Replace with a sentinel (worse than worst rank)
worst_rank = max(df[["b_match_wc_rank", "r_match_wc_rank"]].max().max(), 16) + 1
df["b_match_wc_rank"] = df["b_match_wc_rank"].fillna(worst_rank)
df["r_match_wc_rank"] = df["r_match_wc_rank"].fillna(worst_rank)

print(f"\nShape: {raw.shape} -> {df.shape}")
print(f"Total NaNs remaining: {df.isna().sum().sum()}")


---
## 3. Exploratory Analysis

The fun part. I'm not going to drown the notebook in twenty plots that all say the same thing – just the charts that actually change how I think about the data.


In [ ]:
wc = df["weight_class"].value_counts().reset_index()
wc.columns = ["weight_class", "fights"]

fig = px.bar(wc, x="fights", y="weight_class", orientation="h",
             template=PLOTLY_TEMPLATE, color="fights",
             color_continuous_scale="Reds",
             title="Where the action lives — UFC fights per weight class (2010–2026)")
fig.update_layout(yaxis={"categoryorder": "total ascending"},
                  height=520, coloraxis_showscale=False)
fig.show()


**What I noticed.** Lightweight and Welterweight dominate, which tracks – they've historically been the deepest divisions and the easiest to fill cards with. Women's Featherweight is basically a rounding error, so any modeling we do there will be noisy at best.


In [ ]:
red_rate = df["red_win"].mean()
print(f"Red corner win rate: {red_rate:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Corner bias
sns.barplot(x=["Red corner", "Blue corner"],
            y=[red_rate, 1 - red_rate],
            palette=[RED, BLUE], ax=axes[0])
axes[0].set_ylim(0, 1); axes[0].set_ylabel("Win rate")
axes[0].set_title("Corner bias — it's real, and it's not subtle")
for i, v in enumerate([red_rate, 1 - red_rate]):
    axes[0].text(i, v + 0.015, f"{v:.1%}", ha="center", color="white")

# Red win rate over time
yearly = df.groupby("year")["red_win"].agg(["mean", "count"]).reset_index()
yearly = yearly[yearly["count"] >= 30]
axes[1].plot(yearly["year"], yearly["mean"], marker="o", color=RED, lw=2)
axes[1].axhline(0.5, ls="--", color="#888")
axes[1].set_ylim(0.4, 0.75)
axes[1].set_title("Red corner win rate by year")
axes[1].set_ylabel("Red win %")
plt.tight_layout(); plt.show()


**What I noticed.** Red corner wins ~58% of the time. That's not a bug in the data – the UFC assigns the red corner to the favorite or higher-ranked fighter, so this is a **selection effect**, not magic. Any honest model has to engineer features that are symmetric (or use the diff columns we already have), otherwise it'll learn "predict red" and look smarter than it is.


In [ ]:
# Build a per-fighter career table from the corner stats.
# For each row, both corners contribute their *career-to-date* snapshot.
red_view = df[["red_fighter","red_wins","red_losses","red_wins_by_ko",
               "red_wins_by_submission","red_total_rounds_fought","red_total_title_bouts",
               "red_height_cms","red_reach_cms","red_age","red_stance","fight_date"]].copy()
red_view.columns = ["fighter","wins","losses","wins_ko","wins_sub","total_rounds",
                    "title_bouts","height","reach","age","stance","fight_date"]

blue_view = df[["blue_fighter","blue_wins","blue_losses","blue_wins_by_ko",
                "blue_wins_by_submission","blue_total_rounds_fought","blue_total_title_bouts",
                "blue_height_cms","blue_reach_cms","blue_age","blue_stance","fight_date"]].copy()
blue_view.columns = red_view.columns

long = pd.concat([red_view, blue_view], ignore_index=True)
# Take the *latest* snapshot per fighter as their best career picture
fighters = long.sort_values("fight_date").groupby("fighter").tail(1).copy()
fighters["wins_dec"] = (fighters["wins"] - fighters["wins_ko"] - fighters["wins_sub"]).clip(lower=0)
fighters["total_fights"] = fighters["wins"] + fighters["losses"]
fighters = fighters[fighters["total_fights"] >= 3]  # drop one-and-dones
fighters["ko_rate"]  = fighters["wins_ko"]  / fighters["wins"].replace(0, np.nan)
fighters["sub_rate"] = fighters["wins_sub"] / fighters["wins"].replace(0, np.nan)
fighters["dec_rate"] = fighters["wins_dec"] / fighters["wins"].replace(0, np.nan)
fighters["finish_rate"] = (fighters["wins_ko"] + fighters["wins_sub"]) / fighters["wins"].replace(0, np.nan)
print(f"Tracked fighters: {len(fighters):,}")
fighters.head(3)


In [ ]:
# Distribution of how *wins* break down across the roster
totals = pd.Series({
    "KO/TKO":     fighters["wins_ko"].sum(),
    "Submission": fighters["wins_sub"].sum(),
    "Decision":   fighters["wins_dec"].sum(),
})
totals_pct = totals / totals.sum() * 100

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=totals_pct.index, y=totals_pct.values,
            palette=["#d62728", "#1f77b4", "#aaaaaa"], ax=ax)
for i, v in enumerate(totals_pct.values):
    ax.text(i, v + 0.7, f"{v:.1f}%", ha="center", color="white")
ax.set_ylabel("% of all career wins (roster-wide)")
ax.set_title("How fights actually end (aggregated across careers)")
plt.tight_layout(); plt.show()


**What I noticed.** Decisions are the modal outcome – which lines up with the "the sport got more technical" narrative. It also means a model that's good at picking the *winner* in a 15-minute grind is more valuable than one that can predict knockouts (those are almost stochastic).


In [ ]:
# Stance matchup heatmap - who beats who?
stance_grid = (df.assign(matchup=lambda d: d["red_stance"] + " vs " + d["blue_stance"])
                 .groupby(["red_stance", "blue_stance"])
                 .agg(n=("red_win","size"), red_win_rate=("red_win","mean"))
                 .reset_index())
keep = ["Orthodox", "Southpaw", "Switch"]
stance_grid = stance_grid[stance_grid["red_stance"].isin(keep) &
                          stance_grid["blue_stance"].isin(keep) &
                          (stance_grid["n"] >= 20)]
pivot_wr = stance_grid.pivot(index="red_stance", columns="blue_stance", values="red_win_rate")
pivot_n  = stance_grid.pivot(index="red_stance", columns="blue_stance", values="n")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.heatmap(pivot_wr, annot=pivot_n.astype(int), fmt="d", cmap="RdBu_r",
            center=0.5, vmin=0.45, vmax=0.7, cbar_kws={"label": "Red win rate"},
            linewidths=0.5, linecolor="#0f1116", ax=ax)
ax.set_title("Stance vs Stance — red corner win rate\n(numbers = sample size)")
ax.set_xlabel("Blue stance"); ax.set_ylabel("Red stance")
plt.tight_layout(); plt.show()


**What I noticed.** "Southpaws are tricky" is a real thing in striking circles, but most of the variance here is still red-corner bias bleeding through. The matrix is mostly red because red wins ~58% across every cell. The interesting reading isn't *who's red* – it's that the spread between cells is only a few percentage points. Stance alone is a weak signal.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age distribution
ages = pd.concat([df["red_age"], df["blue_age"]]).dropna()
sns.histplot(ages, bins=40, color=RED, ax=axes[0], edgecolor="none")
axes[0].axvline(ages.median(), ls="--", color="white", alpha=0.7)
axes[0].set_title(f"Fighter ages on fight night (median {ages.median():.0f})")
axes[0].set_xlabel("Age")

# Reach diff
sns.histplot(df["reach_diff"].dropna(), bins=50, color="#f5b041",
             ax=axes[1], edgecolor="none")
axes[1].axvline(0, ls="--", color="white", alpha=0.7)
axes[1].set_title("Reach difference (red − blue, cm)")
axes[1].set_xlabel("cm")

# Height diff
sns.histplot(df["height_diff"].dropna(), bins=50, color="#2ecc71",
             ax=axes[2], edgecolor="none")
axes[2].axvline(0, ls="--", color="white", alpha=0.7)
axes[2].set_title("Height difference (red − blue, cm)")
axes[2].set_xlabel("cm")

plt.tight_layout(); plt.show()


**What I noticed.** The age distribution peaks right around the late 20s – the well-known UFC prime window. Reach and height differences are roughly symmetric around zero, which is exactly what we want for diff features: the model won't be fighting a built-in skew.


In [ ]:
activity = (pd.concat([df["red_fighter"], df["blue_fighter"]])
              .value_counts().head(15)
              .rename_axis("fighter").reset_index(name="ufc_fights"))

fig = px.bar(activity.sort_values("ufc_fights"), x="ufc_fights", y="fighter",
             orientation="h", template=PLOTLY_TEMPLATE,
             color="ufc_fights", color_continuous_scale="OrRd",
             title="Workhorses — most UFC fights in the dataset")
fig.update_layout(height=520, coloraxis_showscale=False)
fig.show()


In [ ]:
# sig_str_diff, td_diff, sub_att_diff are per-minute style differentials
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color, title in zip(
    axes,
    ["sig_str_diff", "td_diff", "sub_att_diff"],
    ["#d62728", "#1f77b4", "#9b59b6"],
    ["Sig. strikes / min (red − blue)",
     "Takedowns / 15 min (red − blue)",
     "Sub attempts / 15 min (red − blue)"]):
    series = df[col].dropna()
    series = series[series.between(series.quantile(0.01), series.quantile(0.99))]
    sns.histplot(series, bins=50, color=color, edgecolor="none", ax=ax)
    ax.axvline(0, ls="--", color="white", alpha=0.7)
    ax.set_title(title); ax.set_xlabel("")

plt.tight_layout(); plt.show()


**What I noticed.** Striking output is the widest distribution – fighters separate themselves on volume more than on takedowns. Sub-attempt differentials are extremely peaky around zero, which makes sense: most fights barely see a submission attempt. That tells me submission threat is more of a *binary* archetype feature than a continuous one.


---
## 4. Style Matchup Analysis

This is the part casual fans get wrong. "He's bigger and undefeated" isn't a prediction, it's a tweet. What actually matters is **whose game gets imposed**. Let's tag fighters by archetype and see how those archetypes match up.


In [ ]:
# Build per-fighter STYLE means using the per-fight diffs:
# if fighter is red, their style stat ~ +diff/2 above mean; if blue, -diff/2.
# This is a rough proxy but it lets us rank fighters by tendency.
def fighter_style_table(df):
    # Per fight, attribute half the diff to red and the negative half to blue,
    # then average per fighter. (Diff columns are red - blue style metrics.)
    parts = []
    for corner, sign in [("red_fighter", +1), ("blue_fighter", -1)]:
        tmp = pd.DataFrame({
            "fighter": df[corner],
            "slpm_proxy": sign * df["sig_str_diff"] / 2,
            "td_proxy":   sign * df["td_diff"] / 2,
            "sub_proxy":  sign * df["sub_att_diff"] / 2,
        })
        parts.append(tmp)
    sty = pd.concat(parts).groupby("fighter").mean()
    return sty

style = fighter_style_table(df)
# Merge style with career snapshot
fighters_full = fighters.set_index("fighter").join(style, how="left").reset_index()
fighters_full = fighters_full.dropna(subset=["slpm_proxy","td_proxy","sub_proxy",
                                             "finish_rate","dec_rate"])
print(f"Fighters with full style profile: {len(fighters_full):,}")


In [ ]:
f = fighters_full.copy()
# Percentile flags (rank within roster)
for col in ["slpm_proxy", "td_proxy", "sub_proxy", "finish_rate", "dec_rate"]:
    f[col + "_p"] = f[col].rank(pct=True)

def tag(row):
    if row["sub_proxy_p"] > 0.85:        return "Submission threat"
    if row["td_proxy_p"]  > 0.80:        return "Grappler / wrestler"
    if row["slpm_proxy_p"] > 0.85:       return "Volume striker"
    if row["finish_rate_p"] > 0.80 and row["wins"] >= 4:
                                         return "Finisher"
    if row["dec_rate_p"] > 0.80 and row["wins"] >= 4:
                                         return "Decision specialist"
    if row["slpm_proxy_p"] > 0.55 and row["td_proxy_p"] < 0.45:
                                         return "Striker"
    return "Well-rounded"

f["archetype"] = f.apply(tag, axis=1)
arch_counts = f["archetype"].value_counts().rename_axis("archetype").reset_index(name="n")
print(arch_counts)

fig = px.bar(arch_counts, x="archetype", y="n", template=PLOTLY_TEMPLATE,
             color="n", color_continuous_scale="Reds",
             title="How the UFC roster splits by style (rough tags)")
fig.update_layout(coloraxis_showscale=False, height=420)
fig.show()


In [ ]:
# Map archetypes back onto each fight
arch_map = dict(zip(f["fighter"], f["archetype"]))
df["red_archetype"]  = df["red_fighter"].map(arch_map)
df["blue_archetype"] = df["blue_fighter"].map(arch_map)

mu = (df.dropna(subset=["red_archetype","blue_archetype"])
        .groupby(["red_archetype","blue_archetype"])
        .agg(n=("red_win","size"), red_wr=("red_win","mean"))
        .reset_index())
mu = mu[mu["n"] >= 25]
order = ["Grappler / wrestler","Submission threat","Volume striker","Striker",
         "Finisher","Decision specialist","Well-rounded"]
pivot = mu.pivot(index="red_archetype", columns="blue_archetype", values="red_wr")
pivot = pivot.reindex(index=[o for o in order if o in pivot.index],
                       columns=[o for o in order if o in pivot.columns])
pivot_n = mu.pivot(index="red_archetype", columns="blue_archetype", values="n").reindex(
    index=pivot.index, columns=pivot.columns)

fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(pivot, annot=pivot_n.fillna(0).astype(int), fmt="d",
            cmap="RdBu_r", center=0.5, vmin=0.35, vmax=0.75,
            cbar_kws={"label":"Red corner win rate"},
            linewidths=0.5, linecolor="#0f1116", ax=ax)
ax.set_title("Archetype vs archetype — red win rate\n(cell labels = sample size)")
ax.set_xlabel("Blue archetype"); ax.set_ylabel("Red archetype")
plt.tight_layout(); plt.show()


**What I noticed.** Once you control for who's tagged what, you can see grapplers tend to *travel well* against most striker archetypes – the classic "wrestling beats everything if it's well-implemented" pattern. Volume strikers eat into pure finishers more than you'd expect, probably because finishers tend to look for one shot and grind to a halt if it doesn't land.

Now the fun query: **which fighters overperform their physical tools?** I'll define overperformance as a fighter whose win rate is well above what their reach/age/size would predict.


In [ ]:
# Each fighter's UFC win % from the per-fight data (more honest than career flag)
fight_records = []
for corner, won in [("red_fighter","red_win"), ("blue_fighter", None)]:
    if won == "red_win":
        tmp = df[[corner,"red_win"]].rename(columns={corner:"fighter","red_win":"won"})
    else:
        tmp = df[[corner,"red_win"]].copy()
        tmp["won"] = 1 - tmp["red_win"]
        tmp = tmp.rename(columns={corner:"fighter"})[["fighter","won"]]
    fight_records.append(tmp)
recs = pd.concat(fight_records).groupby("fighter")["won"].agg(["mean","size"])
recs = recs[recs["size"] >= 6].rename(columns={"mean":"ufc_win_rate","size":"ufc_fights"})

# Predict win rate from physicals only with a quick linear baseline
from sklearn.linear_model import LinearRegression
phys_df = f.set_index("fighter")[["height","reach","age"]].dropna()
joined = recs.join(phys_df, how="inner").dropna()
lr = LinearRegression().fit(joined[["height","reach","age"]], joined["ufc_win_rate"])
joined["expected"] = lr.predict(joined[["height","reach","age"]])
joined["overperf"] = joined["ufc_win_rate"] - joined["expected"]
top_over = joined.sort_values("overperf", ascending=False).head(12).round(3)
top_over[["ufc_fights","ufc_win_rate","expected","overperf","reach","height","age"]]


**What I noticed.** The overperformer list is a rogues' gallery of the fighters whose names sound like ringtones – the kind of guys who shouldn't be winning by the size chart but absolutely do. This is exactly the population a physics-first model will undersell.


---
## 5. Feature Engineering

The dataset hands us most of the diffs we'd want, but I'm going to combine them into a few composite scores that I'd actually use in a write-up:

- **Physical advantage** – reach + height − age (taller, longer, fresher).
- **Striking pressure** – sig-strike differential weighted toward the more active fighter.
- **Grappling pressure** – takedowns and sub attempts, where the difference is the *threat*, not the points.
- **Experience gap** – difference in total rounds + title bouts. A fighter who's been in five-round wars 8 times has a different gas tank than someone with 6 prelim minutes.
- **Finish tendency** – career KO + sub rate. Guys who finish create variance.
- **Stance matchup** – categorical, because Orthodox vs Southpaw is structurally different from Orthodox vs Orthodox.


In [ ]:
def safe_rate(num, den):
    den = den.replace(0, np.nan)
    return (num / den).fillna(0)

m = df.copy()

# Career finish & decision rates per corner
m["red_finish_rate"]  = safe_rate(m["red_wins_by_ko"]  + m["red_wins_by_submission"],  m["red_wins"])
m["blue_finish_rate"] = safe_rate(m["blue_wins_by_ko"] + m["blue_wins_by_submission"], m["blue_wins"])
m["finish_rate_diff"] = m["red_finish_rate"] - m["blue_finish_rate"]

m["red_dec_rate"]  = safe_rate(m["red_wins"]  - m["red_wins_by_ko"]  - m["red_wins_by_submission"],  m["red_wins"])
m["blue_dec_rate"] = safe_rate(m["blue_wins"] - m["blue_wins_by_ko"] - m["blue_wins_by_submission"], m["blue_wins"])
m["dec_rate_diff"] = m["red_dec_rate"] - m["blue_dec_rate"]

# Experience
m["red_total_fights"]  = m["red_wins"]  + m["red_losses"]
m["blue_total_fights"] = m["blue_wins"] + m["blue_losses"]
m["experience_diff"]   = m["red_total_fights"] - m["blue_total_fights"]

# Composite scores - scaled gently
m["physical_adv"]   = (m["reach_diff"].fillna(0) + m["height_diff"].fillna(0)
                       - 1.5 * m["age_diff"].fillna(0))
m["striking_pres"]  = m["sig_str_diff"].fillna(0)
m["grappling_pres"] = m["td_diff"].fillna(0) + 0.5 * m["sub_att_diff"].fillna(0)
m["form_diff"]      = (m["win_streak_diff"].fillna(0)
                       - m["lose_streak_diff"].fillna(0))

# Stance matchup as a categorical
m["stance_matchup"] = m["red_stance"] + "_vs_" + m["blue_stance"]

# Final feature columns
feature_cols = [
    "physical_adv","striking_pres","grappling_pres",
    "experience_diff","form_diff","longest_win_streak_diff",
    "finish_rate_diff","dec_rate_diff",
    "reach_diff","height_diff","age_diff",
    "wins_diff","losses_diff","ko_diff","submission_diff",
    "rank_diff","title_bout_diff","rounds_diff",
    "stance_matchup",
]
X_full = m[feature_cols + ["red_win","fight_date"]].dropna(subset=["red_win"])
print("Modeling frame:", X_full.shape)
X_full.head(3)


---
## 6. Modeling

Quick honesty check before any numbers print: the theoretical ceiling on UFC prediction is probably somewhere in the **65–72% accuracy** range without using betting odds. Sharp Vegas books – with closing-line odds – hover around 67-68%. If our model lands in the low 60s without odds, that's a real result, not a disappointment.

I'm holding out the odds columns deliberately. The point is to see what we can learn from fighter *attributes*, not from the wisdom of the betting market.


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, roc_curve,
                             confusion_matrix)
from sklearn.calibration import calibration_curve

cat = ["stance_matchup"]
num = [c for c in feature_cols if c not in cat]

pre_linear = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])
pre_tree = ColumnTransformer([
    ("num", "passthrough", num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

# Lean configs - fewer/smaller trees, fast on Kaggle CPU.
models = {
    "LogReg": Pipeline([("pre", pre_linear),
                        ("clf", LogisticRegression(max_iter=1000, C=0.5, solver="liblinear"))]),
    "RandomForest": Pipeline([("pre", pre_tree),
                              ("clf", RandomForestClassifier(
                                  n_estimators=200, max_depth=8,
                                  min_samples_leaf=30, n_jobs=-1, random_state=42))]),
    "HistGB": Pipeline([("pre", pre_tree),
                        ("clf", HistGradientBoostingClassifier(
                            max_iter=200, max_depth=5, learning_rate=0.06,
                            early_stopping=True, validation_fraction=0.15,
                            random_state=42))]),
}

try:
    from lightgbm import LGBMClassifier
    models["LightGBM"] = Pipeline([("pre", pre_tree),
        ("clf", LGBMClassifier(n_estimators=300, max_depth=-1, learning_rate=0.05,
                               num_leaves=31, subsample=0.9, colsample_bytree=0.9,
                               random_state=42, n_jobs=-1, verbose=-1))])
except Exception as e:
    print("LightGBM not available:", e)

print("Models:", list(models.keys()))


In [ ]:
data = X_full.sort_values("fight_date").reset_index(drop=True)
X = data[feature_cols]
y = data["red_win"].astype(int)

# Time-respecting split: train on older fights, test on the most recent ~20%
split = int(len(data) * 0.8)
X_tr, X_te = X.iloc[:split], X.iloc[split:]
y_tr, y_te = y.iloc[:split], y.iloc[split:]
print(f"Train: {len(X_tr):,}  Test: {len(X_te):,}  "
      f"(test fights from {data['fight_date'].iloc[split].date()})")

# 3-fold is plenty for ranking models on ~5.8k rows and keeps runtime sane.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
rows, fitted = [], {}
for name, pipe in models.items():
    cv_auc = cross_val_score(pipe, X_tr, y_tr, cv=cv,
                              scoring="roc_auc", n_jobs=-1).mean()
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    rows.append({"model": name,
                 "cv_auc":   round(cv_auc, 3),
                 "test_auc": round(roc_auc_score(y_te, proba), 3),
                 "test_acc": round(accuracy_score(y_te, pred), 3)})
    fitted[name] = pipe

results = pd.DataFrame(rows).sort_values("test_auc", ascending=False).reset_index(drop=True)
results


In [ ]:
best_name = results.iloc[0]["model"]
best_model = fitted[best_name]
best_proba = best_model.predict_proba(X_te)[:, 1]
best_pred  = (best_proba >= 0.5).astype(int)
print(f"Best by test AUC: {best_name}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# ROC curves
for name, pipe in fitted.items():
    p = pipe.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, p)
    axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_te, p):.3f})", lw=2)
axes[0].plot([0,1],[0,1], "--", color="#888")
axes[0].set_title("ROC — every model we trained")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(fontsize=8, loc="lower right")

# Confusion matrix - best
cm = confusion_matrix(y_te, best_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", cbar=False,
            xticklabels=["Blue","Red"], yticklabels=["Blue","Red"], ax=axes[1])
axes[1].set_title(f"Confusion matrix — {best_name}")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")

# Calibration
frac_pos, mean_pred = calibration_curve(y_te, best_proba, n_bins=10, strategy="quantile")
axes[2].plot(mean_pred, frac_pos, marker="o", color=RED, lw=2, label=best_name)
axes[2].plot([0,1],[0,1], "--", color="#888", label="Perfect")
axes[2].set_title("Calibration — are the probs honest?")
axes[2].set_xlabel("Predicted P(red wins)"); axes[2].set_ylabel("Empirical")
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()


**Reading this honestly.** The tree models edge out logistic regression but not by a huge margin – another sign that fight outcomes are heavily noise-driven. The calibration curve is what I actually care about: if the model says "65% Red", it should be right about 65% of the time. Anywhere those points drift far from the diagonal is where I'd take the model's word with a grain of salt.

We're not building a money-printer here. We're building a *plausibility check*.


---
## 7. What's Actually Driving the Predictions?

A score isn't worth much without knowing what the model is leaning on. I want to compare three views: (1) the tree's built-in feature importance, (2) permutation importance (more honest), and (3) SHAP if the install gods cooperate.


In [ ]:
from sklearn.inspection import permutation_importance

# Pick a tree-based winner for importance work
tree_pick = None
for k in ["LightGBM", "HistGB", "RandomForest"]:
    if k in fitted:
        tree_pick = k; break
tree_model = fitted[tree_pick]
print(f"Explainability target: {tree_pick}")

# Feature names after one-hot
oh = tree_model.named_steps["pre"].named_transformers_["cat"]
oh_names = list(oh.get_feature_names_out(cat))
feat_names = num + oh_names

# Built-in importance (RF / LGBM expose feature_importances_; HistGB does not)
clf = tree_model.named_steps["clf"]
imp_df = None
if hasattr(clf, "feature_importances_"):
    imp_df = (pd.DataFrame({"feature": feat_names, "importance": clf.feature_importances_})
                .sort_values("importance", ascending=False).head(15))

# Permutation importance — keep it cheap: subsample test, fewer repeats
rng = np.random.RandomState(42)
sub = rng.choice(len(X_te), size=min(800, len(X_te)), replace=False)
perm = permutation_importance(tree_model, X_te.iloc[sub], y_te.iloc[sub],
                              n_repeats=3, random_state=42,
                              n_jobs=-1, scoring="roc_auc")
perm_df = (pd.DataFrame({"feature": feature_cols,
                         "perm_importance": perm.importances_mean})
             .sort_values("perm_importance", ascending=False).head(15))

if imp_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=imp_df, y="feature", x="importance", color=RED, ax=axes[0])
    axes[0].set_title(f"{tree_pick} — built-in importance")
    sns.barplot(data=perm_df, y="feature", x="perm_importance", color="#f5b041", ax=axes[1])
    axes[1].set_title("Permutation importance on test (Δ AUC)")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=perm_df, y="feature", x="perm_importance", color="#f5b041", ax=ax)
    ax.set_title(f"{tree_pick} — permutation importance on test (Δ AUC)")
plt.tight_layout(); plt.show()


In [ ]:
# SHAP — strictly on a small sample to keep this snappy.
try:
    import shap
    if tree_pick in ("LightGBM", "RandomForest"):
        X_te_enc = tree_model.named_steps["pre"].transform(X_te)
        rng = np.random.RandomState(0)
        idx = rng.choice(X_te_enc.shape[0], size=min(150, X_te_enc.shape[0]), replace=False)
        explainer = shap.TreeExplainer(tree_model.named_steps["clf"])
        sv = explainer.shap_values(X_te_enc[idx])
        if isinstance(sv, list):
            sv = sv[1]
        shap.summary_plot(sv, X_te_enc[idx], feature_names=feat_names,
                          show=True, max_display=12, plot_size=(8, 5))
    else:
        print(f"Skipping SHAP — {tree_pick} doesn't expose a clean TreeExplainer path. "
              "Permutation importance above covers the same story.")
except Exception as e:
    print("SHAP unavailable or failed (that's fine):", e)


**What stands out.** Rank gap, win-streak differential and total-wins differential do the heavy lifting – essentially, *recent dominance* and *career résumé*. The big surprise for casual fans: **raw reach and height barely move the needle**. The broadcast graphic that shows a 4-inch reach advantage doesn't predict the fight; what's been happening in that fighter's last three appearances does. Age difference matters more than physical size, which lines up with the "youth and activity over reach" lesson the sport has been quietly teaching for a decade.


---
## 8. Fight Probability Simulator

The fun toy. Pick two fighters who exist in the dataset, and the function pulls their *most recent* snapshot, builds the diff vector, and returns win probabilities + the matchup factors that pushed the prediction.


In [ ]:
def latest_snapshots(df):
    base = ["stance","height_cms","reach_cms","age","wins","losses",
            "wins_by_ko","wins_by_submission","current_win_streak",
            "current_lose_streak","longest_win_streak","total_rounds_fought",
            "total_title_bouts"]
    red = df[["red_fighter"] + [f"red_{c}" for c in base]
             + ["r_match_wc_rank","fight_date"]].copy()
    red.columns = ["fighter"] + base + ["rank","fight_date"]

    blue = df[["blue_fighter"] + [f"blue_{c}" for c in base]
              + ["b_match_wc_rank","fight_date"]].copy()
    blue.columns = red.columns

    snap = (pd.concat([red, blue], ignore_index=True)
              .sort_values("fight_date")
              .groupby("fighter").tail(1)
              .set_index("fighter"))
    return snap

snap = latest_snapshots(df)
print(f"Fighter snapshots: {len(snap):,}")
snap.head(3)


In [ ]:
_perm_series = pd.Series(perm.importances_mean, index=feature_cols)

def build_diff_row(a, b):
    """a, b: pandas Series of one snapshot each (a is treated as 'red')."""
    a_finish = (a["wins_by_ko"] + a["wins_by_submission"]) / max(a["wins"], 1)
    b_finish = (b["wins_by_ko"] + b["wins_by_submission"]) / max(b["wins"], 1)
    a_dec = (a["wins"] - a["wins_by_ko"] - a["wins_by_submission"]) / max(a["wins"], 1)
    b_dec = (b["wins"] - b["wins_by_ko"] - b["wins_by_submission"]) / max(b["wins"], 1)
    return {
        "physical_adv":     (a["reach_cms"] - b["reach_cms"])
                          + (a["height_cms"] - b["height_cms"])
                          - 1.5 * (a["age"] - b["age"]),
        "striking_pres":    0.0,
        "grappling_pres":   0.0,
        "experience_diff":  (a["wins"]+a["losses"]) - (b["wins"]+b["losses"]),
        "form_diff":        (a["current_win_streak"]-a["current_lose_streak"])
                          - (b["current_win_streak"]-b["current_lose_streak"]),
        "longest_win_streak_diff": a["longest_win_streak"] - b["longest_win_streak"],
        "finish_rate_diff": a_finish - b_finish,
        "dec_rate_diff":    a_dec - b_dec,
        "reach_diff":       a["reach_cms"] - b["reach_cms"],
        "height_diff":      a["height_cms"] - b["height_cms"],
        "age_diff":         a["age"] - b["age"],
        "wins_diff":        a["wins"] - b["wins"],
        "losses_diff":      a["losses"] - b["losses"],
        "ko_diff":          a["wins_by_ko"] - b["wins_by_ko"],
        "submission_diff":  a["wins_by_submission"] - b["wins_by_submission"],
        "rank_diff":        a["rank"] - b["rank"] if pd.notna(a["rank"]) and pd.notna(b["rank"]) else 0,
        "title_bout_diff":  a["total_title_bouts"] - b["total_title_bouts"],
        "rounds_diff":      a["total_rounds_fought"] - b["total_rounds_fought"],
        "stance_matchup":   f"{a['stance']}_vs_{b['stance']}",
    }

def predict_fight(name_a, name_b, model=best_model, snapshots=snap):
    if name_a not in snapshots.index or name_b not in snapshots.index:
        missing = [n for n in (name_a, name_b) if n not in snapshots.index]
        raise ValueError(f"Not in dataset: {missing}")
    a, b = snapshots.loc[name_a], snapshots.loc[name_b]
    row  = pd.DataFrame([build_diff_row(a, b)])[feature_cols]
    p_a  = model.predict_proba(row)[0, 1]

    contrib = row[num].iloc[0].values * _perm_series.reindex(num).values
    contrib_df = (pd.DataFrame({"feature": num, "leans_toward_A": contrib})
                    .reindex(np.argsort(-np.abs(contrib))).head(5))

    print(f"\n{name_a}  vs  {name_b}")
    print(f"  P({name_a} wins) = {p_a:.1%}")
    print(f"  P({name_b} wins) = {1-p_a:.1%}")
    print("  Most influential factors in this matchup:")
    for _, r in contrib_df.iterrows():
        side = name_a if r["leans_toward_A"] > 0 else name_b
        print(f"    - {r['feature']:<24} → leans {side}  ({r['leans_toward_A']:+.3f})")
    return p_a


In [ ]:
# Pick a few real, recent matchups from the dataset
def pick_pair(yr_min=2023):
    sample = df[df["year"] >= yr_min].sample(1, random_state=None).iloc[0]
    return sample["red_fighter"], sample["blue_fighter"]

for a, b in [("Islam Makhachev", "Charles Oliveira"),
             ("Alex Pereira",    "Israel Adesanya"),
             ("Sean O'Malley",   "Merab Dvalishvili")]:
    try:
        predict_fight(a, b)
    except ValueError as e:
        print(e)


---
## 9. Upsets — Where the Model Gets Humbled

The model is just a confidence machine, and the sport laughs at confidence. Let me find the fights on the test set where it was most wrong, and see if there's a pattern.


In [ ]:
test_idx = data.index[split:]
test_meta = m.loc[X_full.index].iloc[split:].copy()
test_meta["pred_red_p"] = best_proba
test_meta["actual_red_win"] = y_te.values

# Upset = model strongly favored the loser
test_meta["model_pick"] = np.where(test_meta["pred_red_p"] >= 0.5, "red", "blue")
test_meta["actual"]     = np.where(test_meta["actual_red_win"] == 1, "red", "blue")
test_meta["model_conf"] = np.where(test_meta["model_pick"] == "red",
                                    test_meta["pred_red_p"],
                                    1 - test_meta["pred_red_p"])
upsets = test_meta[(test_meta["model_pick"] != test_meta["actual"]) &
                   (test_meta["model_conf"] >= 0.65)].copy()
print(f"Strong upsets (model conf ≥ 65% in losing direction): {len(upsets)} of {len(test_meta)} test fights")

biggest = upsets.sort_values("model_conf", ascending=False).head(12)
biggest[["fight_date","red_fighter","blue_fighter","weight_class",
         "pred_red_p","actual","model_conf"]].round(3)


In [ ]:
# Vectorized upset patterns
red_won = upsets["actual_red_win"].values == 1

def winner_against(diff_col):
    d = upsets[diff_col].values
    # True when the winning fighter had the *worse* (smaller) value of this metric
    return ((d < 0) & red_won) | ((d > 0) & ~red_won)

patterns = pd.Series({
    "Shorter fighter won":          winner_against("reach_diff").mean(),
    "Older fighter won":            (((upsets["age_diff"].values > 0) & red_won) |
                                     ((upsets["age_diff"].values < 0) & ~red_won)).mean(),
    "Less experienced fighter won": winner_against("experience_diff").mean(),
    "Lower-ranked fighter won":     (((upsets["rank_diff"].values > 0) & red_won) |
                                     ((upsets["rank_diff"].values < 0) & ~red_won)).mean(),
    "Grappler beat striker":        (((upsets["grappling_pres"].values < 0) & red_won) |
                                     ((upsets["grappling_pres"].values > 0) & ~red_won)).mean(),
}).sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=patterns.values * 100, y=patterns.index, color=RED, ax=ax)
ax.set_xlabel("% of upsets where this was true")
ax.set_title("Anatomy of an upset")
for i, v in enumerate(patterns.values * 100):
    ax.text(v + 0.5, i, f"{v:.0f}%", va="center", color="white")
plt.tight_layout(); plt.show()


**What I noticed.** Most upsets aren't a 5'7" fighter beating a 6'4" fighter – they're a less-experienced or lower-ranked fighter winning a fight the resume says they shouldn't. That's the noise floor of MMA. A guy in his prime, on a great camp, sometimes just *wins*. There is no model feature for "he showed up in the best shape of his life."

---
## 10. Final Takeaways

A few honest conclusions after pushing on this data for a while:

- **Activity and résumé beat physicals.** Streaks, win-rate, rank gap and rounds fought consistently outranked reach and height in importance. The casual fan's first instinct (the bigger guy wins) is mostly wrong.
- **Age matters more than people think.** Even a 4-5 year age gap shows up as a meaningful signal, especially past 35.
- **Stance and weight class are almost noise on their own.** They only matter as interactions with other features.
- **The accuracy ceiling is real.** Without market-derived signals (odds, line movement), you're capped somewhere around the low-to-mid 60s. That isn't a model failure – it's the sport.
- **The "Decision specialist vs Finisher" matchup is genuinely informative**, more than I expected. Style tags carry signal that raw stats miss.

**What I'd add next** to push past this ceiling:
1. **Closing-line betting odds** as a feature (they're already in the dataset – I deliberately held them out).
2. **Recent-form windows**: last 3 fights specifically, not career totals.
3. **Short-notice fight flag** – huge predictor in real life, completely absent here.
4. **Layoff length** since last fight (ring rust + post-loss rebound effects).
5. **Camp / coach changes** – impossible without external scraping, but historically decisive.
6. **Injury reports** – also external, also decisive.
7. **Per-fight method-of-victory** so we can model not just *who* wins but *how*, which is where the real betting edge lives.

If you got this far, thanks for reading. UFC will keep humbling models for as long as the sport exists – the goal isn't to "solve" it, it's to be a little less wrong than the next person at the bar.
